In [1]:
from ecostyles import EcoStyles
# Create styles instance
styles = EcoStyles()

import altair as alt
import pandas as pd
import json
from IPython.display import display

In [2]:
# Register and enable a theme
styles.register_and_enable_theme(theme_name="article")

In [3]:
# Load oilexchange chart spec
with open('oilexchgandfpicharts.json') as file:
    spec = json.load(file)

# 1. Trim "Panel A/B/C" prefix from titles
spec['hconcat'][0]['title'] = 'Brent crude (US$ per barrel)'
spec['hconcat'][1]['title'] = 'Net FPI flows (US$ billion)'
spec['hconcat'][2]['title'] = '₹ per US$ (monthly average)'

# 2/3. Remove gridlines
spec['config']['axisX'].pop('gridColor', None)
spec['config']['axisX']['grid'] = False
spec['config']['axisY'].pop('gridColor', None)
spec['config']['axisY'].pop('gridWidth', None)
spec['config']['axisY']['grid'] = False

# 4. Remove hardcoded colors
# Panel A (Brent)
spec['hconcat'][0]['layer'][1]['mark']['color'] = '#e6224b'
spec['hconcat'][0]['layer'][1]['mark']['point']['color'] = '#e6224b'
# Panel B (FPI)
spec['hconcat'][1]['layer'][2]['mark']['color'] = '#179fdb'
spec['hconcat'][1]['layer'][2]['mark']['point']['color'] = '#179fdb'
# Panel C (rupee)
spec['hconcat'][2]['layer'][1]['mark']['color'] = '#36b7b4'
spec['hconcat'][2]['layer'][1]['mark']['point']['color'] = '#36b7b4'

# 5. Change Panel A from area to line
spec['hconcat'][0]['layer'][1]['mark']['type'] = 'line'
spec['hconcat'][0]['layer'][1]['mark'].pop('opacity', None)

# 6. Move Panel C y-axis to the left, remove y-axis title
spec['hconcat'][2]['layer'][1]['encoding']['y']['axis'].pop('orient', None)
spec['hconcat'][2]['layer'][1]['encoding']['y']['axis'].pop('title', None)

# 7. Remove Panel A y-axis title
spec['hconcat'][0]['layer'][1]['encoding']['y']['axis'].pop('title', None)

# 8. Panel B: Convert millions to billions
spec['hconcat'][1]['transform'] = [{'calculate': 'datum.fpi / 1000', 'as': 'fpi_bn'}]
spec['hconcat'][1]['layer'][2]['encoding']['y']['field'] = 'fpi_bn'
spec['hconcat'][1]['layer'][2]['encoding']['y']['axis'].pop('title', None)
spec['hconcat'][1]['layer'][2]['encoding']['y']['axis']['labelExpr'] = "'$' + format(datum.value, '.1f') + ' bn'"
spec['hconcat'][1]['layer'][2]['encoding']['y']['axis']['values'] = [-12, -8, -4, 0, 4]
spec['hconcat'][1]['layer'][2]['encoding']['y']['scale']['domain'] = [-15, 5]
# Update Panel B tooltip to match billions field
spec['hconcat'][1]['layer'][2]['encoding']['tooltip'][1] = {
    'field': 'fpi_bn',
    'type': 'quantitative',
    'format': ',.2f',
    'title': 'Net FPI (US$ bn)'
}

# Apply theme's config on top
spec.pop('config', None)
theme_config = alt.theme.get()()['config']
spec['config'] = theme_config

# Remove gridlines
for key in spec['config']:
    if key.lower().startswith('axis') and isinstance(spec['config'][key], dict):
        spec['config'][key]['grid'] = False

# 9. Add source
spec['title'] = {
    'text': ['Source: EIA, NSDL, Federal Reserve H.10'],
    'orient': 'bottom',
    'fontStyle': 'italic',
    'fontSize': 10,
    'color': '#676A8680',
    'fontWeight': 'normal',
    'frame': 'group',
    'dx': 0,
    'offset': 7
}

# Render
display({'application/vnd.vegalite.v5+json': spec}, raw=True)

In [4]:
# Set final dimensions
spec['width'] = 450
spec['height'] = 360

# Save chart
with open('oilexchgandfpicharts_styled.json', 'w') as f:
    json.dump(spec, f, indent=2)

In [5]:
# Load inrexchange chart spec
with open('inrexchng.json') as file:
    spec = json.load(file)

# 1. Convert hardcoded colors to theme palette
# Main INR/USD line
spec['layer'][2]['mark']['color'] = '#36b7b4'
# Episode shading
spec['layer'][0]['encoding']['color']['value'] = '#F4C245'
# Vertical event lines
spec['layer'][1]['mark']['color'] = '#676A86'
# Highlight dots (2022 peak / 2026 record)
spec['layer'][4]['encoding']['color'] = {'value': '#E6224B'}

# 2. Move y-axis to left
spec['layer'][2]['encoding']['y']['axis'].pop('orient', None)

# 3. Shorten x-axis labels to just year
spec['layer'][2]['encoding']['x']['axis']['values'] = ["2022-01-01", "2023-01-01", "2024-01-01", "2025-01-01", "2026-01-01"]
spec['layer'][2]['encoding']['x']['axis']['format'] = "%Y"

# 4. Fix cutoff label — widen x domain slightly on both ends so Dec 2022 label
spec['layer'][2]['encoding']['x']['scale']['domain'] = ["2021-10-01", "2026-08-01"]
spec['layer'][0]['encoding']['x']['scale']['domain'] = ["2021-10-01", "2026-08-01"]
spec['width'] = 450

# 5. Shift both text labels to right (away from dotted lines / dots)
spec['layer'][5]['mark']['align'] = 'left'
spec['layer'][5]['mark']['dx'] = -53
spec['layer'][5]['mark']['dy'] = -13

# 6. Add source label
spec['title'] = {
    'text': ['Source: Federal Reserve H.10'],
    'orient': 'bottom',
    'fontStyle': 'italic',
    'fontSize': 10,
    'color': '#676A8680',
    'fontWeight': 'normal',
    'frame': 'group',
    'dx': 0,
    'offset': 7
}

# Fix date format bug
spec['layer'][5]['data']['values'][1]['observation_date'] = "2026-05-01"

# Apply your theme's config on top (font, etc.)
spec.pop('config', None)
theme_config = alt.theme.get()()['config']
spec['config'] = theme_config
for key in spec['config']:
    if key.lower().startswith('axis') and isinstance(spec['config'][key], dict):
        spec['config'][key]['grid'] = False

display({'application/vnd.vegalite.v5+json': spec}, raw=True)

In [6]:
# Set final dimensions
spec['width'] = 450
spec['height'] = 360

# Save chart
with open('inrexchng_styled.json', 'w') as f:
    json.dump(spec, f, indent=2)